In [21]:
import requests
import json

# Define the API endpoint URL
api_url = "https://nileva.meta-bytes.net/api/products"

# Make GET request to the endpoint
response = requests.get(api_url)

# Check if request was successful
if response.status_code == 200:
    # Parse JSON response
    products_data = response.json()
    
    # Save data to file for later use
    with open('products.json', 'w') as f:
        json.dump(products_data, f, indent=4)
        
    print(f"Successfully retrieved {len(products_data)} products")
    print("Data saved to products.json")
else:
    print(f"Failed to get data: {response.status_code}")
    print(response.text)


Successfully retrieved 3 products
Data saved to products.json


In [22]:
import pandas as pd

# print(response.text)

In [23]:
response.json()['data']['products']

[{'product_id': 1,
  'name': 'Herforte Lotion Spray (to fill in gaps and treat alopecia)',
  'description': 'Hair forte is a mixture of natural extracts and extracts of the most important hair oils with anti-dandruff agent, to be a quick treatment for hair loss, guaranteed and tested by dermatologists for fortifying hair & increase hair shaft strength to prevent hair loss. Also, Hair forte spray stimulates hair germination to be an effective solution to fill the frontal spaces in women and men, in addition to treating dandruff & itching associated with dryness and inflammation of the scalp, Hair forte spray is suitable for all hair types even oily & dry scalp.\r\n\r\nComposition:-\r\nHair forte spray with its unique formula to be effective in stimulating the blood circulation of the scalp and hair follicles to strengthen and nourish them to prevent hair loss and also to stimulate root germination for the growth of healthy, thick, long and strong hair without dandruff.\r\n\r\n•\tJojoba 

In [24]:
df = pd.DataFrame(response.json()['data']['products'])

In [25]:
concern = 'skin'

df[df['concerns'].apply(
        lambda x: any(concern.lower() in c[0].lower() for c in x if c)
    )].to_dict('records')

[{'product_id': 3,
  'name': 'Valdo moisturizing cream',
  'description': 'Valdo Moisturizing Cream\r\n-The best moisturizing cream for the face - hands - body and sensitive areas -\r\n -Valdo treats severe dryness and rebuilds the skin barrier.\r\n- Makes the skin soft and hydrated all day without scales\r\n-Valdo cream is rich in anti-inflammatory and antibacterial.\r\n -To treat skin infections and dissections also sunburn and burns first degree -\r\n-Accelerates wound healing. It is worth noting that its composition Valdo cream is rich in antioxidants.\r\n  - Collagen stimulator to provide freshness and get rid of pallor and dullness.\r\n - Protection against the appearance of premature wrinkles\r\n- For all skin types (for dry skin - for oily skin - for combination and normal skin)\r\n\r\nComposition:\r\nValdo cream is an ideal natural mix from most important natural extracts, oils& vitamins for complete skin care\r\n•\tPanthenol “Vitamin B5”.\r\n•\tAloe Vera.\r\n•\tCalendula.\r\n

In [28]:
df

,product_id,name,description,ingredients,concerns,category,price,product_file
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,[],[[Hair loss]],"[Hair Care, Hair spray& lotion]",200.00,None
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,[],[[Dandruff]],"[Hair Care, Shampoos]",150.00,None
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream\r\n-The best moisturi...,[],[[Eczema & Dry skin]],"[Skin Care, Moisturizer]",80.00,None
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,[],"[[Strawberry skin treatment, Acne& acne spots,...","[Personal Care, Soft peel and Exfoliant]",100.00,None
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,[],"[[Hyperpigmentation& dark spots, Sensitive are...","[Skin Care, Whitening cream]",100.00,None
5,6,Spleet Eye Contour Cream (for treating dark ci...,An unique potent mix of natural ingredients...,[],"[[Hyperpigmentation& dark spots, Dark circles ...","[Skin Care, Eye cream]",225.00,None
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,[],"[[Acne& acne spots, Hyperpigmentation& dark sp...","[Skin Care, Cleansers]",220.00,None
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,[],[[Sensitive areas& Bikini area brightening]],"[Personal Care, Cleansers]",90.00,None
8,14,"Valdo Sunscreen Lotion (No Oxidation, No Sweat...",الحمايةمن االلتهاباتوجفاف البشرة.\r\nيحمي من...,[],[[Eczema & Dry skin]],"[Skin Care, Sun screen]",350.00,None
9,15,Hairforte Oil Replacement Cream (for treating ...,Hairfort cream is a special formula composed...,[],[[Dandruff]],"[Hair Care, Oil replacement]",180.00,None


In [29]:
# Convert price column to float
df['price'] = df['price'].astype(float)

# Display DataFrame info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    24 non-null     int64  
 1   name          24 non-null     object 
 2   description   24 non-null     object 
 3   ingredients   24 non-null     object 
 4   concerns      24 non-null     object 
 5   category      24 non-null     object 
 6   price         24 non-null     float64
 7   product_file  0 non-null      object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1.6+ KB


In [9]:
from typing import List, Dict, Optional
from langchain_core.tools import tool
import pandas as pd

# Assuming your DataFrame is loaded as df
# df = pd.read_csv('your_products.csv')

@tool
def search_products_by_concern(concern: str) -> List[Dict]:
    """
    Search for products by health/beauty concern (e.g., 'Hair loss', 'Dandruff', 'Eczema').
    
    Args:
        concern (str): The health or beauty concern to search for
    
    Returns:
        List[Dict]: A list of products addressing the specified concern
    """
    matches = df[df['concerns'].apply(
        lambda x: any(concern.lower() in c[0].lower() for c in x if c)
    )]
    return matches.to_dict('records')

@tool
def search_products_by_category(category: str) -> List[Dict]:
    """
    Search for products by category (e.g., 'Hair Care', 'Skin Care').
    
    Args:
        category (str): The category to search for
    
    Returns:
        List[Dict]: A list of products in the specified category
    """
    matches = df[df['category'].apply(
        lambda x: any(category.lower() in c.lower() for c in x)
    )]
    return matches.to_dict('records')

@tool
def search_products_by_name(name: str) -> List[Dict]:
    """
    Search for products by name.
    
    Args:
        name (str): The product name to search for (partial matches allowed)
    
    Returns:
        List[Dict]: A list of products matching the name query
    """
    matches = df[df['name'].str.lower().str.contains(name.lower())]
    return matches.to_dict('records')

@tool
def get_lowest_price_products(n: int = 5) -> List[Dict]:
    """
    Get the lowest priced products.
    
    Args:
        n (int): Number of products to return
    
    Returns:
        List[Dict]: A list of the n lowest priced products
    """
    return df.nsmallest(n, 'price').to_dict('records')

@tool
def get_highest_price_products(n: int = 5) -> List[Dict]:
    """
    Get the highest priced products.
    
    Args:
        n (int): Number of products to return
    
    Returns:
        List[Dict]: A list of the n highest priced products
    """
    return df.nlargest(n, 'price').to_dict('records')

@tool
def get_products_in_price_range(min_price: float, max_price: float) -> List[Dict]:
    """
    Get products within a specific price range.
    
    Args:
        min_price (float): Minimum price
        max_price (float): Maximum price
    
    Returns:
        List[Dict]: List of products within the specified price range
    """
    matches = df[(df['price'] >= min_price) & (df['price'] <= max_price)]
    return matches.to_dict('records')

# List of tools to be used with LangGraph
tools = [
    search_products_by_concern,
    search_products_by_category,
    search_products_by_name,
    get_lowest_price_products,
    get_highest_price_products,
    get_products_in_price_range
]

In [ ]:
search_products_by_concern()